# Notebook 1 — Data Cleaning
### CDC Foodborne Disease Outbreak Surveillance System (1998–2015)

**Goal:** Create an analysis-ready dataset without conflating unreported severity outcomes with observed zeros.
**Input:** `data/outbreaks.csv` — 19,119 source records × 12 columns
**Output:** `data/cleaned_data.csv` — 18,839 retained records × 9 columns

**Steps:**
1. Audit the original data, missingness and full-row duplicates.
2. Remove only 280 identical original 12-column records (row equality does not prove outbreak identity).
3. Drop the three unused fields after deduplication, preserving 11 records previously collapsed by projection.
4. Fill missing categorical fields with `Unknown` while preserving unreported Hospitalizations/Fatalities as `NaN`.
5. Validate source alignment, missingness and export to `data/`.

**Important:** Rows are surveillance records, not independently verified unique outbreak events. The 11 restored rows may or may not represent separate outbreaks; event-level identifiers are unavailable.

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np

df_raw = pd.read_csv("data/outbreaks.csv")
print(f"Shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
df_raw.head(3)

## 2. Initial audit

Understanding what we have before making any changes.

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Missingness — absolute counts and percentage
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)
audit = pd.DataFrame({"missing_n": missing, "missing_%": missing_pct,
                       "unique_values": df_raw.nunique()})
audit[audit["missing_n"] > 0].sort_values("missing_%", ascending=False)

In [ ]:
# Duplicate check
n_dupes = df_raw.duplicated().sum()
print(f"Duplicate rows: {n_dupes} ({n_dupes/len(df_raw)*100:.1f}% of dataset)")

**Audit summary and missingness policy**

| Column | Decision |
|---|---|
| `Ingredient`, `Serotype/Genotype`, `Status` | Retain during duplicate detection; drop afterward (not used for current EDA) |
| `Food`, `Species`, `Location` | Missing category → `Unknown` |
| `Hospitalizations`, `Fatalities` | Missing outcome → retain `NaN` (not equivalent to an observed zero) |

Original full-row duplicates: **280**. Removing columns first creates **291** projected duplicates, which would remove 11 additional records differing in discarded fields. This notebook preserves those records. Neither method verifies event identity.

Do not calculate severity rates over illness counts from outbreaks with unreported severity outcomes; the observed denominator should be outcome-specific and reporting coverage should be disclosed.

## 3. Remove exact duplicate original records

In [ ]:
# Remove exact original-row duplicates BEFORE discarding distinguishing columns.
# This does not establish whether identical records describe the same real-world outbreak.
original_rows = len(df_raw)
original_duplicate_count = int(df_raw.duplicated().sum())
assert original_rows == 19119, f"Unexpected raw record count: {original_rows}"
assert original_duplicate_count == 280, (
    f"Original duplicate policy needs review: {original_duplicate_count}"
)
df = df_raw.drop_duplicates().copy()
assert len(df) == 18839, f"Unexpected retained records: {len(df)}"
print(f"Exact 12-column row duplicates removed: {original_duplicate_count:,}")
print(f"Retained source records: {len(df):,}")

## 4. Drop non-analytical columns AFTER duplicate detection

In [ ]:
# Discard non-analytical fields only after full-source deduplication.
# Distinct original records must not be merged by projecting onto nine columns.
excluded_columns = ["Ingredient", "Serotype/Genotype", "Status"]
projected_duplicate_count = int(df_raw.drop(columns=excluded_columns).duplicated().sum())
assert projected_duplicate_count == 291, (
    f"Unexpected legacy projected duplicate count: {projected_duplicate_count}"
)
df = df.drop(columns=excluded_columns).reset_index(drop=True)
print(f"Legacy projected duplicate count (NOT used for removal): {projected_duplicate_count}")
print(f"Extra source records retained vs legacy workflow: {projected_duplicate_count - original_duplicate_count}")
print(f"Columns after drop: {df.columns.tolist()}")

## 5. Handle missing values without zero-imputing unreported severity

In [ ]:
# Categorical columns → 'Unknown'
for col in ["Location", "Food", "Species"]:
    n_filled = df[col].isna().sum()
    df[col] = df[col].fillna("Unknown")
    print(f"{col}: filled {n_filled:,} missing values → 'Unknown'")

In [ ]:
# Keep numerical outcome missingness explicit. An unreported outcome is not a zero.
# pandas NaN is written as an empty CSV field and restored on read_csv.
for col in ["Hospitalizations", "Fatalities"]:
    observed = int(df[col].notna().sum())
    unreported = int(df[col].isna().sum())
    print(f"{col}: {observed:,} observed records; {unreported:,} unreported (NaN preserved)")
# Other numeric columns are not imputed.

## 6. Final validation (categorical completeness, outcome missingness)

In [ ]:
# Validate categories while allowing genuine unreported severity outcomes.
assert len(df) == 18839 and len(df.columns) == 9
assert not df[["Location", "Food", "Species"]].isna().any().any()
assert df[["Hospitalizations", "Fatalities"]].isna().sum().to_dict() == {
    "Hospitalizations": 3560, "Fatalities": 3538
}, "Unexpected missingness: inspect original records or duplicate policy"
assert (df["Illnesses"] > 0).all()
print(f"Validated: {len(df):,} retained records × {df.shape[1]} fields")
print("Unreported outcomes preserved as NaN; observed zeros remain zero.")
print(df[["Hospitalizations", "Fatalities"]].isna().sum())

In [ ]:
# Sanity check: key column distributions
print("Year range:  ", df["Year"].min(), "–", df["Year"].max())
print("States:      ", df["State"].nunique(), "unique values")
print("Food:        ", df["Food"].nunique(), "unique values (incl. 'Unknown')")
print("Species:     ", df["Species"].nunique(), "unique values (incl. 'Unknown')")
df.describe()

## 7. Export

In [ ]:
from pathlib import Path
# Run from repository root. Write to the data/ path read by NB2, NB3, NB4 and dashboard.
output_path = Path("data") / "cleaned_data.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(output_path, index=False)
print(f"Saved {output_path}: {len(df):,} records × {len(df.columns)} columns")
print("NB2 must preserve the two numerical outcome NaNs when constructing grouped data.")
print("Do not interpret saved grouped/EDA/dashboard metrics as updated until stage 2 and 3.")